# SmolLM2-135M Memory Fusion — Sequential Acceptance Training v3

This fixes the startup failure caused by the repository's generic mandatory Hugging Face backup hook. This experiment already stores accepted layers, the current in-progress layer, logs, and final checkpoints on **Google Drive**, so an HF token is no longer required when the output directory is under `/content/drive/`.

The training algorithm is unchanged: one Transformer attention layer is replaced at a time, trained against the Transformer teacher, checked on real student hidden states, and accepted only after the functional and model-level gates pass.


In [1]:
import os, sys, pathlib, subprocess, json, time
import torch

subprocess.run(['nvidia-smi'], check=False)
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if not REPO_DIR.exists():
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'], check=True)

subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR),'transformers==4.57.6','datasets>=3,<5','huggingface_hub>=0.34,<2','pandas','matplotlib'], check=True)
print('python:', sys.executable)
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())


python: /usr/bin/python3
torch: 2.11.0+cu128 cuda: True


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/TinyCeNN-LM')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('persistent root:', DRIVE_ROOT)


Mounted at /content/drive
persistent root: /content/drive/MyDrive/TinyCeNN-LM


## Settings
Keep these defaults for the first run. `RESUME=True` recovers accepted layers and the current partially trained layer from Drive.


In [3]:
BASE_MODEL='HuggingFaceTB/SmolLM2-135M'
MEMORY_RANK=64
FEATURE_DIM=32
CONTEXT_LENGTH=128
SEED=73
RESUME=True
STRICT_ACCEPTANCE=True
MIN_LAYER_STEPS=50
MAX_LAYER_STEPS=300
CHECK_EVERY=25
LAYER_LR=2e-4
TEACHER_ALPHA_START=0.90
TEACHER_ALPHA_END=0.0
ACCEPT_NMSE=0.20
ACCEPT_COSINE=0.90
ACCEPT_INCREMENTAL_DELTA_NLL=0.015
ACCEPT_CUMULATIVE_DELTA_NLL=0.05
CORE_O_TOKENS=50_000
NORM_TOKENS=50_000
FULL_TOKENS=100_000
MAX_RUNTIME_MINUTES=240
OUTPUT_DIR=DRIVE_ROOT/f'smollm2-memory-fusion-sequential-r{MEMORY_RANK}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('output:', OUTPUT_DIR)


output: /content/drive/MyDrive/TinyCeNN-LM/smollm2-memory-fusion-sequential-r64


## Train / resume
The launcher detects that the output is on Google Drive and skips only the redundant mandatory HF backup. The actual sequential checkpoints remain mandatory on Drive. Child output is streamed live and copied to `last_colab_run_v3.log`.


In [4]:
cmd=[sys.executable,'-u',str(REPO_DIR/'scripts'/'train_smollm2_memory_fusion_sequential_v3.py'),'--base-model',BASE_MODEL,'--output-dir',str(OUTPUT_DIR),'--memory-rank',str(MEMORY_RANK),'--feature-dim',str(FEATURE_DIM),'--context-length',str(CONTEXT_LENGTH),'--seed',str(SEED),'--min-layer-steps',str(MIN_LAYER_STEPS),'--max-layer-steps',str(MAX_LAYER_STEPS),'--check-every',str(CHECK_EVERY),'--layer-lr',str(LAYER_LR),'--teacher-alpha-start',str(TEACHER_ALPHA_START),'--teacher-alpha-end',str(TEACHER_ALPHA_END),'--accept-nmse',str(ACCEPT_NMSE),'--accept-cosine',str(ACCEPT_COSINE),'--accept-incremental-delta-nll',str(ACCEPT_INCREMENTAL_DELTA_NLL),'--accept-cumulative-delta-nll',str(ACCEPT_CUMULATIVE_DELTA_NLL),'--core-o-tokens',str(CORE_O_TOKENS),'--norm-tokens',str(NORM_TOKENS),'--full-tokens',str(FULL_TOKENS),'--max-runtime-minutes',str(MAX_RUNTIME_MINUTES)]
cmd.append('--resume' if RESUME else '--no-resume')
cmd.append('--strict-acceptance' if STRICT_ACCEPTANCE else '--no-strict-acceptance')
log_path=OUTPUT_DIR/'last_colab_run_v3.log'
print(' '.join(cmd))
print('Log:', log_path)
env=os.environ.copy()
env['PYTHONUNBUFFERED']='1'
with log_path.open('w',encoding='utf-8') as log:
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    assert p.stdout is not None
    for line in p.stdout:
        print(line,end='')
        log.write(line); log.flush()
    rc=p.wait()
print('Process return code:',rc)
if rc!=0:
    err=OUTPUT_DIR/'sequential_last_error.json'
    if err.exists():
        print('--- SAVED SOFTWARE ERROR ---')
        print(err.read_text())
    raise RuntimeError(f'Sequential trainer failed with return code {rc}. See {log_path}.')


/usr/bin/python3 -u /content/TinyCeNN-LM/scripts/train_smollm2_memory_fusion_sequential_v3.py --base-model HuggingFaceTB/SmolLM2-135M --output-dir /content/drive/MyDrive/TinyCeNN-LM/smollm2-memory-fusion-sequential-r64 --memory-rank 64 --feature-dim 32 --context-length 128 --seed 73 --min-layer-steps 50 --max-layer-steps 300 --check-every 25 --layer-lr 0.0002 --teacher-alpha-start 0.9 --teacher-alpha-end 0.0 --accept-nmse 0.2 --accept-cosine 0.9 --accept-incremental-delta-nll 0.015 --accept-cumulative-delta-nll 0.05 --core-o-tokens 50000 --norm-tokens 50000 --full-tokens 100000 --max-runtime-minutes 240 --resume --strict-acceptance
Log: /content/drive/MyDrive/TinyCeNN-LM/smollm2-memory-fusion-sequential-r64/last_colab_run_v3.log
[TinyCeNN][PERSISTENCE] Google Drive output detected; using Drive checkpoints and skipping redundant mandatory HF backup.
[TinyCeNN][PROCESS START] train_smollm2_memory_fusion_sequential_v2
[TinyCeNN][BACKUP] parent mandatory backup detected; child fallback not

## Current persistent status
Run this after training pauses or finishes. A status of `current_layer_needs_more_training` is **not a crash**; rerun the training cell to continue the same saved layer.


In [5]:
status_path=OUTPUT_DIR/'sequential_run_status.json'
progress_path=OUTPUT_DIR/'sequential_progress.json'
in_progress_path=OUTPUT_DIR/'sequential_in_progress.json'
for p in [status_path,progress_path,in_progress_path]:
    if p.exists():
        print('\n###',p.name)
        data=json.loads(p.read_text())
        print(json.dumps(data,indent=2)[:12000])

if progress_path.exists():
    progress=json.loads(progress_path.read_text())
    print('\nAccepted layers:',len(progress.get('accepted_layers',[])),progress.get('accepted_layers',[]))
if in_progress_path.exists():
    cur=json.loads(in_progress_path.read_text())
    print('Current layer:',cur.get('current_layer'),'rounds completed:',cur.get('rounds_completed'))



### sequential_run_status.json
{
  "status": "current_layer_needs_more_training",
  "accepted_layers": [
    0
  ],
  "current_layer": 1,
  "rounds_completed": 4,
  "last_report": {
    "layer": 1,
    "accepted": false,
    "steps": 250,
    "nmse": 0.0732264518737793,
    "cosine": 0.9634184837341309,
    "probe_nll": 2.9952849745750427,
    "incremental_delta_nll": 0.06823426485061646,
    "cumulative_delta_nll": 0.08219295740127563
  },
  "message": "No next Transformer layer was replaced. Rerun to continue this same layer."
}

### sequential_progress.json
{
  "format_version": 1,
  "stage": "accepted_layer_0",
  "accepted_layers": [
    0
  ],
  "config": {
    "feature_dim": 32,
    "memory_rank": 64,
    "dilations": [
      1,
      2,
      4,
      8,
      16,
      32,
      64,
      128
    ],
    "shifted_window": 8,
    "train_output_projection": true
  },
  "layer_reports": [
    {
      "layer": 0,
      "accepted": false,
      "steps": 275,
      "nmse": 0.04809959